# Deep Hedging: Learning Optimal Hedging Strategies

## Introduction

This notebook demonstrates **deep hedging** - using neural networks to learn optimal hedging strategies that outperform traditional delta hedging when:

- **Transaction costs** are significant
- **Discrete rebalancing** is required
- **Market frictions** exist (bid-ask spread, liquidity constraints)
- **Risk preferences** are non-standard (CVaR, asymmetric loss)

---

### What is Deep Hedging?

Traditional delta hedging:
```
Hedge position δ = ∂V/∂S  (Black-Scholes delta)
```

Deep hedging learns a policy `π(state) → hedge position` that **minimises a risk measure** over the terminal P&L distribution.

### Why Does This Matter?

| Traditional Hedging | Deep Hedging |
|---------------------|-------------|
| Assumes no costs | Learns cost-optimal rebalancing |
| Continuous rebalancing | Discrete, realistic |
| Risk-neutral | Any risk measure (VaR, CVaR) |
| Model-dependent | Can be model-free |

---

### References

- Bühler et al. (2019) "Deep Hedging" - Quantitative Finance
- Cao et al. (2021) "Deep Hedging of Long-Term Financial Derivatives"

In [ ]:
# =============================================================================
# SETUP: Imports
# =============================================================================

import sys
from pathlib import Path
import numpy as np
from datetime import date

sys.path.insert(0, str(Path.cwd().parents[1]))

# Plotting
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# QuantStrata deep hedging components
from src.deep_hedging.core.types import HedgingState, DeepHedgingTrainingConfig
from src.deep_hedging.core.risk_measures import MeanVarianceRisk, CVaR
from src.deep_hedging.core.costs import ProportionalCost
from src.deep_hedging.environments.gbm import GBMHedgingEnv
from src.deep_hedging.agents.deep import DeepHedgingAgent, MLPPolicy
from src.deep_hedging.agents.delta import DeltaHedgingAgent

print("All imports successful!")

## 1. Setting Up the Hedging Environment

We simulate a hedging scenario:
- **Option**: Short a European call option (we need to hedge!)
- **Underlying**: GBM dynamics (for simplicity)
- **Costs**: Proportional transaction costs

In [ ]:
# =============================================================================
# STEP 1: Define Hedging Environment
# =============================================================================

# Option parameters
S0 = 100.0      # Initial spot
K = 100.0       # Strike (ATM)
T = 0.25        # 3 months to expiry
r = 0.05        # Risk-free rate
sigma = 0.20    # Volatility (20%)

# Simulation parameters
n_steps = 63    # Daily hedging for ~3 months
dt = T / n_steps

# Transaction cost (10 bps per trade)
cost_rate = 0.001
cost_model = ProportionalCost(rate=cost_rate)

# Create hedging environment
env = GBMHedgingEnv(
    S0=S0,
    K=K,
    T=T,
    r=r,
    sigma=sigma,
    n_steps=n_steps,
    option_type="call",
    position="short",  # We're short the option, need to hedge
    cost_model=cost_model,
)

print("Hedging Environment:")
print("="*50)
print(f"  Position:     Short Call")
print(f"  Spot:         {S0}")
print(f"  Strike:       {K}")
print(f"  Expiry:       {T} years ({int(T*252)} trading days)")
print(f"  Volatility:   {sigma:.0%}")
print(f"  Risk-free:    {r:.0%}")
print(f"  Hedging freq: {n_steps} times")
print(f"  Cost:         {cost_rate:.2%} per trade")

## 2. Benchmark: Delta Hedging

First, let's see how traditional **Black-Scholes delta hedging** performs.

Delta hedging:
1. Compute BSM delta at each step
2. Rebalance hedge to match delta
3. Accumulate trading costs

In [ ]:
# =============================================================================
# STEP 2: Delta Hedging Benchmark
# =============================================================================

# Create delta hedging agent
delta_agent = DeltaHedgingAgent()

# Simulate many paths
n_paths = 10000
np.random.seed(42)

delta_pnls = []
delta_costs = []

for i in range(n_paths):
    # Reset environment (generates new price path)
    state = env.reset()
    
    total_cost = 0.0
    prev_hedge = 0.0
    
    # Run hedging episode
    while not env.done:
        # Get delta hedge
        hedge = delta_agent.act(state)
        
        # Trade cost
        trade_size = abs(hedge - prev_hedge)
        cost = cost_model.compute(trade_size, state.spot)
        total_cost += cost
        prev_hedge = hedge
        
        # Step environment
        state, _ = env.step(hedge)
    
    # Terminal P&L (option payoff - hedging P&L - costs)
    terminal_pnl = env.get_terminal_pnl()
    delta_pnls.append(terminal_pnl)
    delta_costs.append(total_cost)

delta_pnls = np.array(delta_pnls)
delta_costs = np.array(delta_costs)

print("\nDelta Hedging Results (10,000 paths):")
print("="*50)
print(f"  Mean P&L:           ${delta_pnls.mean():.2f}")
print(f"  Std P&L:            ${delta_pnls.std():.2f}")
print(f"  Mean Costs:         ${delta_costs.mean():.2f}")
print(f"  5% VaR:             ${np.percentile(-delta_pnls, 95):.2f}")
print(f"  95% CVaR:           ${-delta_pnls[delta_pnls < np.percentile(delta_pnls, 5)].mean():.2f}")

## 3. Deep Hedging Agent

Now let's train a **neural network** to learn a better hedging policy.

The agent learns:
- When to rebalance (not always!)
- How much to hedge (may differ from delta)
- To minimize a risk measure (not just variance)

In [ ]:
# =============================================================================
# STEP 3: Create Deep Hedging Agent
# =============================================================================

# Policy network: MLP that maps state -> hedge position
# Input features: [log_moneyness, time_to_maturity, current_position, running_pnl]
input_dim = 4
hidden_dims = [32, 32]  # 2 hidden layers

# Risk measure: Mean-Variance with risk aversion
risk_measure = MeanVarianceRisk(risk_aversion=2.0)

# Create agent
deep_agent = DeepHedgingAgent(
    input_dim=input_dim,
    hidden_dims=hidden_dims,
    risk_measure=risk_measure,
    learning_rate=0.001,
    activation='tanh',
)

print("\nDeep Hedging Agent:")
print("="*50)
print(f"  Input features:    {input_dim}")
print(f"  Hidden layers:     {hidden_dims}")
print(f"  Risk measure:      Mean-Variance (λ=2.0)")
print(f"  Total parameters:  ~{sum(d*hidden_dims[0] for d in [input_dim] + hidden_dims) + hidden_dims[-1]}")

In [ ]:
# =============================================================================
# STEP 4: Train the Deep Hedging Agent
# =============================================================================

print("\nTraining Deep Hedging Agent...")
print("="*50)

# Training configuration
training_config = DeepHedgingTrainingConfig(
    n_epochs=100,
    batch_size=256,
    n_paths_per_batch=256,
)

# Training loop
training_history = []

for epoch in range(training_config.n_epochs):
    # Generate batch of paths
    batch_pnls = []
    
    for _ in range(training_config.batch_size):
        state = env.reset()
        
        while not env.done:
            hedge = deep_agent.act(state)
            state, _ = env.step(hedge)
        
        batch_pnls.append(env.get_terminal_pnl())
    
    batch_pnls = np.array(batch_pnls)
    
    # Compute loss (risk measure of negative P&L)
    loss = risk_measure.compute(-batch_pnls)
    training_history.append(loss)
    
    # Update agent (simplified - actual implementation uses proper gradients)
    deep_agent.update(batch_pnls)
    
    if (epoch + 1) % 20 == 0:
        print(f"  Epoch {epoch+1:3d}: Risk = {loss:.4f}, Mean P&L = ${batch_pnls.mean():.2f}")

print("\nTraining complete!")

In [ ]:
# =============================================================================
# STEP 5: Evaluate Deep Hedging Agent
# =============================================================================

# Evaluate on test paths
deep_pnls = []
deep_costs = []

for i in range(n_paths):
    state = env.reset()
    
    total_cost = 0.0
    prev_hedge = 0.0
    
    while not env.done:
        hedge = deep_agent.act(state)
        
        trade_size = abs(hedge - prev_hedge)
        cost = cost_model.compute(trade_size, state.spot)
        total_cost += cost
        prev_hedge = hedge
        
        state, _ = env.step(hedge)
    
    terminal_pnl = env.get_terminal_pnl()
    deep_pnls.append(terminal_pnl)
    deep_costs.append(total_cost)

deep_pnls = np.array(deep_pnls)
deep_costs = np.array(deep_costs)

print("\nDeep Hedging Results (10,000 paths):")
print("="*50)
print(f"  Mean P&L:           ${deep_pnls.mean():.2f}")
print(f"  Std P&L:            ${deep_pnls.std():.2f}")
print(f"  Mean Costs:         ${deep_costs.mean():.2f}")
print(f"  5% VaR:             ${np.percentile(-deep_pnls, 95):.2f}")
print(f"  95% CVaR:           ${-deep_pnls[deep_pnls < np.percentile(deep_pnls, 5)].mean():.2f}")

## 4. Comparison: Delta vs Deep Hedging

In [ ]:
# =============================================================================
# STEP 6: Compare Results
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: P&L distributions
ax1 = axes[0, 0]
ax1.hist(delta_pnls, bins=50, alpha=0.6, label='Delta Hedging', color='blue', density=True)
ax1.hist(deep_pnls, bins=50, alpha=0.6, label='Deep Hedging', color='orange', density=True)
ax1.axvline(x=0, color='black', linestyle='--', linewidth=1)
ax1.axvline(x=delta_pnls.mean(), color='blue', linestyle='-', linewidth=2, label=f'Delta Mean: ${delta_pnls.mean():.1f}')
ax1.axvline(x=deep_pnls.mean(), color='orange', linestyle='-', linewidth=2, label=f'Deep Mean: ${deep_pnls.mean():.1f}')
ax1.set_xlabel('Terminal P&L ($)')
ax1.set_ylabel('Density')
ax1.set_title('P&L Distribution Comparison', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Trading costs
ax2 = axes[0, 1]
ax2.hist(delta_costs, bins=50, alpha=0.6, label='Delta Hedging', color='blue', density=True)
ax2.hist(deep_costs, bins=50, alpha=0.6, label='Deep Hedging', color='orange', density=True)
ax2.axvline(x=delta_costs.mean(), color='blue', linestyle='-', linewidth=2)
ax2.axvline(x=deep_costs.mean(), color='orange', linestyle='-', linewidth=2)
ax2.set_xlabel('Total Trading Costs ($)')
ax2.set_ylabel('Density')
ax2.set_title('Trading Cost Distribution', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Training curve
ax3 = axes[1, 0]
ax3.plot(training_history, 'b-', linewidth=1)
ax3.set_xlabel('Epoch')
ax3.set_ylabel('Risk Measure')
ax3.set_title('Deep Hedging Training Curve', fontweight='bold')
ax3.grid(True, alpha=0.3)

# Plot 4: Summary comparison
ax4 = axes[1, 1]
metrics = ['Mean P&L', 'Std P&L', 'Mean Cost', '5% VaR']
delta_vals = [
    delta_pnls.mean(),
    delta_pnls.std(),
    delta_costs.mean(),
    np.percentile(-delta_pnls, 95),
]
deep_vals = [
    deep_pnls.mean(),
    deep_pnls.std(),
    deep_costs.mean(),
    np.percentile(-deep_pnls, 95),
]

x = np.arange(len(metrics))
width = 0.35

ax4.bar(x - width/2, delta_vals, width, label='Delta', color='blue', alpha=0.7)
ax4.bar(x + width/2, deep_vals, width, label='Deep', color='orange', alpha=0.7)
ax4.set_xticks(x)
ax4.set_xticklabels(metrics)
ax4.set_ylabel('Value ($)')
ax4.set_title('Metric Comparison', fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Summary Comparison
# =============================================================================

print("\n" + "="*70)
print("PERFORMANCE COMPARISON")
print("="*70)
print(f"\n{'Metric':<25} {'Delta Hedging':>15} {'Deep Hedging':>15} {'Improvement':>12}")
print("-"*70)

# Mean P&L (higher is better)
delta_mean = delta_pnls.mean()
deep_mean = deep_pnls.mean()
print(f"{'Mean P&L':<25} ${delta_mean:>13.2f} ${deep_mean:>13.2f} {(deep_mean - delta_mean):>+11.2f}")

# Std P&L (lower is better)
delta_std = delta_pnls.std()
deep_std = deep_pnls.std()
print(f"{'Std P&L':<25} ${delta_std:>13.2f} ${deep_std:>13.2f} {(delta_std - deep_std):>+11.2f}")

# Mean Costs (lower is better)
delta_cost = delta_costs.mean()
deep_cost = deep_costs.mean()
print(f"{'Mean Trading Costs':<25} ${delta_cost:>13.2f} ${deep_cost:>13.2f} {(delta_cost - deep_cost):>+11.2f}")

# VaR (lower is better)
delta_var = np.percentile(-delta_pnls, 95)
deep_var = np.percentile(-deep_pnls, 95)
print(f"{'5% VaR':<25} ${delta_var:>13.2f} ${deep_var:>13.2f} {(delta_var - deep_var):>+11.2f}")

# Sharpe-like ratio
delta_sharpe = delta_mean / delta_std if delta_std > 0 else 0
deep_sharpe = deep_mean / deep_std if deep_std > 0 else 0
print(f"{'Risk-adjusted (mean/std)':<25} {delta_sharpe:>14.3f} {deep_sharpe:>14.3f} {(deep_sharpe - delta_sharpe):>+11.3f}")

print("-"*70)

## 5. Key Takeaways

### When Deep Hedging Helps
- **High transaction costs**: Learns to trade less frequently
- **Illiquid markets**: Adapts to market impact
- **Non-Gaussian returns**: Handles fat tails and jumps
- **Path-dependent options**: No analytic delta exists

### Implementation Considerations
- **Training data**: Need realistic price paths (historical or simulated)
- **Risk measure**: Must match actual risk preferences
- **Model risk**: Agent may overfit to training distribution
- **Interpretability**: Black-box nature can be a concern

### Production Deployment
1. Train offline on historical scenarios
2. Validate against held-out periods
3. Deploy with human oversight
4. Monitor for distribution shift